## 一.环境导入

In [35]:
import torch
print(torch.cuda.is_available())
import numpy as np

True


## 二.张量初始化

### 1.直接从数据创建

In [36]:
data = [[1,2],[3,4]]
x_data = torch.tensor(data)
x_data

tensor([[1, 2],
        [3, 4]])

### 2.从NumPy数组创建

In [37]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
x_np

tensor([[1, 2],
        [3, 4]])

### 3.从另一个张量创建

In [38]:
x_ones = torch.ones_like(x_data) #保留原张量x_data的属性,不只是保留 shape,连 dtype 都一起保留了
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) #创建新张量,形状和 x_data 相同,但用 dtype 参数把数据类型覆盖成 float32
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.5090, 0.0707],
        [0.3859, 0.3727]]) 



### 4.使用随机值或常数值

 `shape` 是一个元组，代表张量的维度。在下面的函数中，它决定了输出张量的维度。<br>
单元素元组的坑(最可能的源头):(2) 是整数 2,不是元组;只有 (2,) 才是元组。<br>
所以, 这里末尾也加上了一个逗号

In [39]:
shape = (2, 3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.3358, 0.0851, 0.7119],
        [0.3671, 0.6345, 0.4631]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


## 三.张量属性
<br>
张量属性描述了它们的形状(shape)、数据类型(dtype)以及存储它们的设备(device)。

In [40]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}\n")
print(f"Datatype of tensor: {tensor.dtype}\n")
print(f"Device tensor is stored on: {tensor.device}\n")

Shape of tensor: torch.Size([3, 4])

Datatype of tensor: torch.float32

Device tensor is stored on: cpu



注意到这里显示"cpu", 张量并不是默认放在gpu上的, 因为gpu的显存非常紧张, 很容易空间不足<br>
`torch.cuda.is_available()` 是"gpu可用"的意思,不是"已经将数据放入gpu内"。进去的指令是 `.to("cuda")`。

## 四.张量运算

### 1.张量的移动

在前面已经提过, 张量的默认存储位置是在cpu(或者说,是搭载cpu的这个机器上,其更精确的位置应该在内存上).<br>
而如果要加速矩阵计算, 应该将张量的物理位置移动至gpu(也就是显存上)

In [41]:
# 如果可以, 我们一般将张量移动至gpu上
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
tensor = tensor.to(device)

print(f"Device tensor is stored on: {tensor.device}")

Device tensor is stored on: cuda:0


`torch.accelerator.current_accelerator().type` 其实是由**三段**拼起来的,每段各有各的用途:

| 段                       | 是什么                 | 干了啥                                                |
| ------------------------ | ---------------------- | ----------------------------------------------------- |
| `torch.accelerator`      | 一个模块(命名空间)     | PyTorch 新加的"加速器管理器",管所有 GPU 类型的设备    |
| `.current_accelerator()` | **方法调用**(注意括号) | "我现在正在用哪个设备?"→ 返回一个 `torch.device` 对象 |
| `.type`                  | **属性**(没括号)       | 那个设备对象的"类型"字段 → 返回字符串 `"cuda"`        |

**`cuda:0` 指的是"第 0 块显卡",不是"第 0 个 CUDA 计算单元"**

### 2.类似numpy的标准索引与切片

In [42]:
tensor = torch.ones(4,4)
tensor[:,1] = 0
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


### 3.连接张量

#### <1>.`tensor.cat()`张量拼接

In [43]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


#### <2>.`tensor.stack()`张量堆叠

In [44]:
t2 = torch.stack([tensor,tensor],dim=0)
print(t2)

tensor([[[1., 0., 1., 1.],
         [1., 0., 1., 1.],
         [1., 0., 1., 1.],
         [1., 0., 1., 1.]],

        [[1., 0., 1., 1.],
         [1., 0., 1., 1.],
         [1., 0., 1., 1.],
         [1., 0., 1., 1.]]])


#### <3>.关于`tensor.cat()`和`tensor.stack()`的区别<br>
| PyTorch       | NumPy            | 行为                  |
| ------------- | ---------------- | --------------------- |
| `torch.cat`   | `np.concatenate` | 沿现有轴拼接,维数不变 |
| `torch.stack` | `np.stack`       | 新加一维打包,维数 +1  |
---

> 大概就是`.cat`相当于把两张纸拼起来,而`.stack`要把两张规模完全一样的纸堆起来

### 4.张量乘法

#### <1>.`tensor.mul()`和`*`(哈达玛积)

In [45]:
# 这计算的是元素级乘积(element-wise product)
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)} \n")
# 可替代的其他语法:
print(f"tensor * tensor \n {tensor * tensor}")

tensor.mul(tensor) 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor * tensor 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


通过上述计算不难发现, `tensor.mul()`和`*`计算的是矩阵上的每个位置上的对应乘积
| 写法               | 数学名称     | 行为                    |
| ------------------ | ------------ | ----------------------- |
| `*` 或 `.mul()`    | **哈达玛积** | 逐位置对应相乘          |
| `@` 或 `.matmul()` | **矩阵乘法** | 行×列的经典线性代数乘法 |
| `.dot()`           | 点积         | 两个**一维向量**的内积  |
---
**哈达玛积**:
- 不需要同 size,广播机制(Broadcasting)会自动处理.<br>
- 注意它不是"补全/填充",准确说是"拉伸对齐"——把小的那个张量概念上拉长成和大的一样,再逐位置相乘。<br>
- 它不会真的复制数据(省内存,只是逻辑上假装拉长了)。<br>
---
**广播机制**:
两个张量从**最右边(末尾)往左**逐维对齐,每一对维度要么:
- **相等** ✓
- **有一个是 1**(拉伸到另一个的尺寸)✓
- **一个没有这个维度**(视作 1)✓
- 都不满足(比如 4 对 3)→ **报错** ❌

#### <2>.`tensor.matmul`和`@`(矩阵乘积)

In [46]:
print(f"tensor.matmul(tensor.T) \n {tensor.matmul(tensor.T)} \n")
# 可替代的语法:
print(f"tensor @ tensor.T \n {tensor @ tensor.T}")

tensor.matmul(tensor.T) 
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]]) 

tensor @ tensor.T 
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])


### 5.原地操作(In-place operations)

后缀为 `_` 的操作是原地操作。<br>
例如：`x.copy_(y)`，`x.t_()` 会直接改变 `x` 的值

In [47]:
print(tensor, "\n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


> WARNING:<br>
> 原地操作可以节省内存，但在计算导数时可能会出现问题，因为会立即丢失历史记录。<br>
> 因此，不建议使用它们。<br>

## 五.与 NumPy 的桥接
CPU 上的张量和 NumPy 数组可以共享底层内存位置，改变其中一个也会改变另一个。

### 1.张量影响 NumPy 数组

In [48]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


张量的变化会反映在 NumPy 数组中.<br>
例如下方, 当t+=1之后, n也+=1了.<br>
这说明两者本质上是在同一物理位置上, 没有复制.<br>

In [49]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


### 2.NumPy 数组影响张量

In [50]:
n = np.ones(5)
t = torch.from_numpy(n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
n: [1. 1. 1. 1. 1.]


同理, NumPy 数组的变化会反映在张量中, 如下:<br>

In [51]:
np.add(n, 1, out=n)  # 对numpy数组n执行+1操作, 并输出至原来的n
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


### 3.关于numpy和tensor的默认浮点精度

"1."输出tensor没有标明`dtype=torch.float64`, 而"2."标明了, 是因为tensor从numpy转换时自动使用双精度吗?
> `torch.from_numpy()` 的规则是:**numpy 是什么 dtype,就原样搬过来,绝不自己改。**<br>
---
为什么第二次会标明`dtype=torch.float64`?
> 这里涉及到numpy和tensor的默认精度标准
> | 默认浮点精度 | 精度类型            |
> | ------------ | ------------------- |
> | numpy        | **float64**(双精度) |
> | torch        | **float32**(单精度) |
> ```
> 在numpy.ones()默认使用了双精度构造, 而由于tensor是原样转变, 所以还是双精度
---
那为什么tensor输出时要**特地**标明是`tensor.float64`?
> 因为和tensor的默认精度标准不符, 所以需要标明
---
为什么tensor默认精度是`float32`呢?
> 消费级显卡对 float64 的支持是"阉割"过的,算起来比 float32 慢一个数量级(约 1/32)。<br>
> 而且 float64 占的内存是 float32 的两倍。<br>

> 所以真正训练,一定会反复看到这个操作:<br>
> 把 numpy 数据喂给模型前,先转成 float32
> ```python
> n = np.load(...)            # 默认 float64
> t = torch.from_numpy(n).float()   # ← 显式转成 float32,GPU 才快
> ```